# PulsePlate RAG / Insight Release Gates Notebook

This notebook implements an offline weekly evaluation lane for PulsePlate RAG/Insight quality gates.

It is aligned with the real repository boundaries:

- `core.ai.prepare_insight_runtime(...)` owns runtime/provider/transparency preparation.
- `app.services.insight_runtime.generate_traced_insight(...)` is the app-layer tracing adapter.
- `core.rag.orchestration.retrieve_and_validate_rag(...)` owns retrieval → validation → prompt construction.
- `core.rag.contracts.RAGChunk` / `RAGContext` define the retrieval contract.
- `app.security.agent_input_guard` must run before RAG/provider/tool execution.
- `core.insight.philosophy_validator.validate_llm_output(...)` is used as a deterministic wellness-safe output check when available.

Default mode is safe and offline: TF-IDF retriever + extractive grounded generator.  
Set `RETRIEVER_MODE=pulseplate` and/or `GENERATOR_MODE=pulseplate_runtime` only inside a local PulsePlate checkout with dependencies and env configured.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
import sys
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Sequence

import numpy as np
import pandas as pd

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except ImportError as exc:
    raise RuntimeError(
        "Install scikit-learn to run the local fallback retriever: "
        "pip install scikit-learn pandas numpy"
    ) from exc

# Safe defaults for local PulsePlate imports.
os.environ.setdefault("TESTING", "true")
os.environ.setdefault("SERVER_SALT", "pulseplate-rag-eval-local-dummy-salt")

PROJECT_ROOT = Path(os.getenv("PULSEPLATE_REPO_ROOT", Path.cwd())).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

EXPERIMENT_ID = os.getenv(
    "EXPERIMENT_ID",
    f"rag_eval_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}",
)

EVAL_INPUT_PATH = Path(
    os.getenv(
        "PULSEPLATE_RAG_EVAL_INPUT",
        str(PROJECT_ROOT / "data" / "evals" / "rag_weekly_500.jsonl"),
    )
).resolve()

ARTIFACT_ROOT = Path(
    os.getenv(
        "PULSEPLATE_RAG_EVAL_ARTIFACT_ROOT",
        str(PROJECT_ROOT / "artifacts" / "rag_eval"),
    )
).resolve()
RUN_DIR = ARTIFACT_ROOT / EXPERIMENT_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_SIZE = int(os.getenv("PULSEPLATE_RAG_EVAL_SAMPLE_SIZE", "500"))
TOP_K = int(os.getenv("PULSEPLATE_RAG_EVAL_TOP_K", "50"))
RANDOM_SEED = int(os.getenv("PULSEPLATE_RAG_EVAL_RANDOM_SEED", "42"))

RETRIEVER_MODE = os.getenv("RETRIEVER_MODE", "local_tfidf").strip().lower()
GENERATOR_MODE = os.getenv("GENERATOR_MODE", "extractive_stub").strip().lower()

ENABLE_NLI_MODEL = os.getenv("ENABLE_NLI_MODEL", "false").strip().lower() in {
    "1",
    "true",
    "yes",
    "on",
}
NLI_MODEL_NAME = os.getenv("NLI_MODEL_NAME", "roberta-large-mnli")

# Release-gate thresholds from your requested plan.
GATE_THRESHOLDS = {
    "evidence_exact_match_rate": float(os.getenv("GATE_EVIDENCE_EXACT_MATCH", "0.70")),
    "mean_nli_entailment": float(os.getenv("GATE_MEAN_NLI_ENTAILMENT", "0.85")),
    "support_precision": float(os.getenv("GATE_SUPPORT_PRECISION", "0.70")),
    "ece": float(os.getenv("GATE_ECE", "0.08")),
    "escalation_min": float(os.getenv("GATE_ESCALATION_MIN", "0.10")),
    "escalation_max": float(os.getenv("GATE_ESCALATION_MAX", "0.25")),
    "recall_at_50": float(os.getenv("GATE_RECALL_AT_50", "0.80")),
}

ROUTING_CONFIDENCE_THRESHOLD = float(os.getenv("ROUTING_CONFIDENCE_THRESHOLD", "0.65"))
SUPPORT_ENTAILMENT_THRESHOLD = float(os.getenv("SUPPORT_ENTAILMENT_THRESHOLD", "0.50"))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("EVAL_INPUT_PATH:", EVAL_INPUT_PATH)
print("RUN_DIR:", RUN_DIR)
print("RETRIEVER_MODE:", RETRIEVER_MODE)
print("GENERATOR_MODE:", GENERATOR_MODE)


## 1. Optional PulsePlate imports

The notebook is defensive: real PulsePlate imports are used when available. If a dependency is missing, the notebook falls back to offline adapters instead of weakening the evaluation contract.


In [ ]:
PULSEPLATE_IMPORTS: dict[str, Any] = {}
PULSEPLATE_IMPORT_ERRORS: dict[str, str] = {}

def _try_import(name: str, import_fn):
    try:
        PULSEPLATE_IMPORTS[name] = import_fn()
    except Exception as exc:
        PULSEPLATE_IMPORTS[name] = None
        PULSEPLATE_IMPORT_ERRORS[name] = f"{type(exc).__name__}: {exc}"

_try_import(
    "scan_ai_agent_input",
    lambda: __import__(
        "app.security.agent_input_guard",
        fromlist=["scan_ai_agent_input"],
    ).scan_ai_agent_input,
)
_try_import(
    "require_safe_ai_agent_input",
    lambda: __import__(
        "app.security.agent_input_guard",
        fromlist=["require_safe_ai_agent_input"],
    ).require_safe_ai_agent_input,
)
_try_import(
    "validate_llm_output",
    lambda: __import__(
        "core.insight.philosophy_validator",
        fromlist=["validate_llm_output"],
    ).validate_llm_output,
)
_try_import(
    "retrieve_and_validate_rag",
    lambda: __import__(
        "core.rag.orchestration",
        fromlist=["retrieve_and_validate_rag"],
    ).retrieve_and_validate_rag,
)
_try_import(
    "prepare_insight_runtime",
    lambda: __import__("core.ai", fromlist=["prepare_insight_runtime"]).prepare_insight_runtime,
)
_try_import(
    "generate_traced_insight",
    lambda: __import__(
        "app.services.insight_runtime",
        fromlist=["generate_traced_insight"],
    ).generate_traced_insight,
)

print("PulsePlate import status:")
for name, value in PULSEPLATE_IMPORTS.items():
    print(f"  {name}: {'OK' if value is not None else 'unavailable'}")
if PULSEPLATE_IMPORT_ERRORS:
    print("\nImport errors:")
    for name, err in PULSEPLATE_IMPORT_ERRORS.items():
        print(f"  {name}: {err}")


## 2. Evaluation input

Expected JSONL/CSV columns:

| Column | Required | Meaning |
|---|---:|---|
| `query_id` | no | Stable ID; generated if absent |
| `query_text` | yes | User-like query |
| `gold_doc_ids` | strongly recommended | JSON/list/string of expected doc IDs or path fragments |
| `gold_answer` | recommended | Reference answer for RAGAS/context recall |
| `expected_claims` | optional | Claims expected or relevant to answer checking |
| `evidence_quotes` | optional | Exact snippets that should appear in retrieved sources/answer |
| `user_tier` | optional | FREE/PRO/VIP routing metadata |
| `subject_id` | optional | Tenant/user ID for real PulsePlate vector retrieval |
| `human_label_if_any` | optional | 1 correct / 0 incorrect for calibration |


In [ ]:
def _json_or_list(value: Any) -> list[str]:
    if value is None:
        return []
    if isinstance(value, float) and math.isnan(value):
        return []
    if isinstance(value, list):
        return [str(x) for x in value if str(x).strip()]
    if isinstance(value, tuple):
        return [str(x) for x in value if str(x).strip()]
    if isinstance(value, str):
        s = value.strip()
        if not s:
            return []
        try:
            parsed = json.loads(s)
            if isinstance(parsed, list):
                return [str(x) for x in parsed if str(x).strip()]
            if isinstance(parsed, str):
                return [parsed]
        except json.JSONDecodeError:
            pass
        if "|" in s:
            return [part.strip() for part in s.split("|") if part.strip()]
        if "," in s and not s.startswith("docs/"):
            return [part.strip() for part in s.split(",") if part.strip()]
        return [s]
    return [str(value)]

def load_eval_input(path: Path) -> pd.DataFrame:
    if not path.exists():
        smoke_rows = [
            {
                "query_id": "local_smoke_001",
                "query_text": "Какие tiers являются каноническими в PulsePlate?",
                "gold_doc_ids": ["AGENTS.md"],
                "gold_answer": "Канонические tiers — FREE, PRO и VIP.",
                "expected_claims": ["PulsePlate canonical tiers are FREE, PRO, and VIP."],
                "evidence_quotes": ["FREE", "PRO", "VIP"],
                "user_tier": "PRO",
                "subject_id": None,
                "human_label_if_any": None,
            },
            {
                "query_id": "local_smoke_002",
                "query_text": "Что делает RAG orchestration layer?",
                "gold_doc_ids": ["core/rag/orchestration.py"],
                "gold_answer": "RAG orchestration retrieves chunks, validates them, and builds a formatted prompt.",
                "expected_claims": ["RAG orchestration retrieves chunks and builds a formatted prompt."],
                "evidence_quotes": ["formatted prompt"],
                "user_tier": "PRO",
                "subject_id": None,
                "human_label_if_any": None,
            },
        ]
        print(f"Input file not found. Using {len(smoke_rows)} embedded smoke rows.")
        return pd.DataFrame(smoke_rows)

    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        rows = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        df = pd.DataFrame(rows)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    elif path.suffix.lower() in {".parquet", ".pq"}:
        df = pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported input extension: {path.suffix}")

    return df

eval_df = load_eval_input(EVAL_INPUT_PATH)

if "query_text" not in eval_df.columns:
    raise ValueError("Evaluation input must contain query_text")

if "query_id" not in eval_df.columns:
    eval_df["query_id"] = [f"q_{i:05d}" for i in range(len(eval_df))]

for col in ["gold_doc_ids", "expected_claims", "evidence_quotes"]:
    if col not in eval_df.columns:
        eval_df[col] = [[] for _ in range(len(eval_df))]
    eval_df[col] = eval_df[col].apply(_json_or_list)

if "gold_answer" not in eval_df.columns:
    eval_df["gold_answer"] = ""
if "user_tier" not in eval_df.columns:
    eval_df["user_tier"] = "PRO"
if "subject_id" not in eval_df.columns:
    eval_df["subject_id"] = None
if "human_label_if_any" not in eval_df.columns:
    eval_df["human_label_if_any"] = None

if len(eval_df) > SAMPLE_SIZE:
    eval_df = eval_df.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)

print("Evaluation rows:", len(eval_df))
display(eval_df.head(5))


## 3. Local corpus and fallback retriever

The fallback corpus is built from `docs/`, `core/`, `app/`, `tests/guards/`, and selected contract-like files. This makes the notebook useful even before a production vector store is available.


In [ ]:
@dataclass(frozen=True)
class CorpusChunk:
    doc_id: str
    source_url: str
    text: str

EXCLUDED_DIRS = {
    ".git",
    ".venv",
    "venv",
    "node_modules",
    "dist",
    "build",
    ".pytest_cache",
    ".mypy_cache",
    ".ruff_cache",
    "artifacts",
    "worktrees",
}

INCLUDED_SUFFIXES = {".md", ".py", ".json", ".yaml", ".yml", ".txt"}

DEFAULT_CORPUS_DIRS = [
    "docs",
    "core",
    "app",
    "tests/guards",
    "AGENTS.md",
    "RUNBOOK_AGENT.md",
]

def _is_excluded(path: Path) -> bool:
    parts = set(path.parts)
    return bool(parts & EXCLUDED_DIRS)

def _read_text(path: Path) -> str:
    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return ""

def chunk_text(text: str, *, max_chars: int = 1200, overlap: int = 160) -> list[str]:
    cleaned = re.sub(r"\n{3,}", "\n\n", text).strip()
    if not cleaned:
        return []
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", cleaned) if p.strip()]
    chunks: list[str] = []
    current = ""
    for para in paragraphs:
        if len(current) + len(para) + 2 <= max_chars:
            current = f"{current}\n\n{para}".strip()
            continue
        if current:
            chunks.append(current)
        if len(para) <= max_chars:
            current = para
        else:
            start = 0
            while start < len(para):
                end = min(start + max_chars, len(para))
                chunks.append(para[start:end])
                if end >= len(para):
                    break
                start = max(0, end - overlap)
            current = ""
    if current:
        chunks.append(current)
    return chunks

def build_local_corpus(project_root: Path) -> list[CorpusChunk]:
    files: list[Path] = []
    for item in DEFAULT_CORPUS_DIRS:
        p = project_root / item
        if p.is_file() and p.suffix.lower() in INCLUDED_SUFFIXES:
            files.append(p)
        elif p.is_dir():
            for child in p.rglob("*"):
                if child.is_file() and child.suffix.lower() in INCLUDED_SUFFIXES and not _is_excluded(child):
                    files.append(child)

    chunks: list[CorpusChunk] = []
    for path in sorted(set(files)):
        rel = path.relative_to(project_root).as_posix()
        text = _read_text(path)
        for idx, chunk in enumerate(chunk_text(text)):
            chunks.append(
                CorpusChunk(
                    doc_id=f"{rel}#chunk-{idx + 1}",
                    source_url=rel,
                    text=chunk,
                )
            )

    if not chunks:
        # Last-resort corpus so smoke execution does not crash outside a checkout.
        fallback = [
            CorpusChunk(
                doc_id="fallback::pulseplate_tiers",
                source_url="fallback",
                text="PulsePlate canonical product tiers are FREE, PRO, and VIP. Premium is deprecated.",
            ),
            CorpusChunk(
                doc_id="fallback::rag_orchestration",
                source_url="fallback",
                text="RAG orchestration retrieves chunks, validates them, and builds a formatted prompt.",
            ),
        ]
        return fallback
    return chunks

corpus = build_local_corpus(PROJECT_ROOT)
print("Corpus chunks:", len(corpus))
print("First corpus item:", asdict(corpus[0]))


In [ ]:
class LocalTfidfRetriever:
    def __init__(self, chunks: Sequence[CorpusChunk]) -> None:
        self.chunks = list(chunks)
        self.vectorizer = TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            max_features=80_000,
        )
        self.matrix = self.vectorizer.fit_transform([c.text for c in self.chunks])

    def retrieve(self, query: str, *, top_k: int) -> list[dict[str, Any]]:
        q = self.vectorizer.transform([query])
        scores = cosine_similarity(q, self.matrix).ravel()
        order = np.argsort(scores)[::-1][:top_k]
        results: list[dict[str, Any]] = []
        for rank, idx in enumerate(order, start=1):
            chunk = self.chunks[int(idx)]
            score = float(scores[int(idx)])
            results.append(
                {
                    "rank": rank,
                    "doc_id": chunk.doc_id,
                    "source_url": chunk.source_url,
                    "retrieval_score": score,
                    "doc_snippet": chunk.text[:1400],
                    "retriever": "local_tfidf",
                }
            )
        return results

local_retriever = LocalTfidfRetriever(corpus)

async def pulseplate_retrieve(query: str, *, top_k: int, subject_id: int | None, context_compaction_enabled: bool | None = None) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    if context_compaction_enabled is None:
        context_compaction_enabled = os.getenv("FEATURE_RAG_CONTEXT_COMPACTION", "false").strip().lower() in {"1", "true", "on", "yes"}
    elif type(context_compaction_enabled) is not bool:
        raise TypeError("context_compaction_enabled must be a built-in bool")

    retrieve_and_validate_rag = PULSEPLATE_IMPORTS.get("retrieve_and_validate_rag")
    if retrieve_and_validate_rag is None:
        raise RuntimeError("PulsePlate retrieve_and_validate_rag is unavailable")
    result = await retrieve_and_validate_rag(
        query,
        max_chunks=top_k,
        philo_validation_enabled=True,
        recursive_rag_enabled=os.getenv("FEATURE_RAG_RECURSIVE", "false").lower() in {"1", "true", "on", "yes"},
        optimization_enabled=os.getenv("FEATURE_RAG_RECURSIVE_OPTIMIZATION", "false").lower() in {"1", "true", "on", "yes"},
        context_compaction_enabled=context_compaction_enabled,
        subject_id=subject_id,
    )
    chunks_compacted = getattr(result, "chunks_compacted", 0)
    if type(chunks_compacted) is not int or chunks_compacted < 0:
        chunks_compacted = 0
    context_compaction_result_observed = (
        context_compaction_enabled is True
        and getattr(result, "context_compaction_completed", False) is True
    )
    if not context_compaction_result_observed:
        chunks_compacted = 0
    retrieved: list[dict[str, Any]] = []
    for rank, chunk in enumerate(getattr(result, "chunks", [])[:top_k], start=1):
        file_name = str(getattr(chunk, "file", "unknown"))
        chunk_id = str(getattr(chunk, "chunk_id", f"rank_{rank}"))
        retrieved.append(
            {
                "rank": rank,
                "doc_id": f"{file_name}::{chunk_id}",
                "source_url": file_name,
                "retrieval_score": float(getattr(chunk, "score", 0.0) or 0.0),
                "doc_snippet": str(getattr(chunk, "content", ""))[:1400],
                "retriever": "pulseplate_retrieve_and_validate_rag",
                "rag_hops": getattr(result, "hops", None),
                "rag_latency_ms": getattr(result, "latency_ms", None),
                "rag_confidence": getattr(result, "confidence", None),
                "rag_warnings": getattr(result, "warnings", []),
                "rag_context_compaction_enabled": context_compaction_enabled,
                "rag_context_compaction_result_observed": context_compaction_result_observed,
                "rag_chunks_compacted": chunks_compacted,
            }
        )
    return retrieved, {
        "context_compaction_enabled": context_compaction_enabled,
        "context_compaction_result_observed": context_compaction_result_observed,
        "chunks_compacted": chunks_compacted,
    }

async def retrieve(query: str, *, top_k: int, subject_id: int | None) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    retrieval_stats = {
        "context_compaction_enabled": False,
        "context_compaction_result_observed": False,
        "chunks_compacted": 0,
    }
    if RETRIEVER_MODE == "pulseplate":
        context_compaction_enabled = os.getenv("FEATURE_RAG_CONTEXT_COMPACTION", "false").strip().lower() in {"1", "true", "on", "yes"}
        retrieval_stats["context_compaction_enabled"] = context_compaction_enabled
        try:
            retrieved, retrieval_stats = await pulseplate_retrieve(
                query,
                top_k=top_k,
                subject_id=subject_id,
                context_compaction_enabled=context_compaction_enabled,
            )
            if retrieved:
                return retrieved, retrieval_stats
        except Exception as exc:
            print(f"[WARN] PulsePlate retriever failed; falling back to local TF-IDF: {type(exc).__name__}: {exc}")
    return local_retriever.retrieve(query, top_k=top_k), retrieval_stats

# Smoke test
_smoke, _smoke_retrieval_stats = await retrieve("PulsePlate FREE PRO VIP tiers", top_k=3, subject_id=None)
pd.DataFrame(_smoke)


## 4. Generator adapters

Default generator is deliberately extractive: it quotes retrieved evidence and avoids external provider calls. This makes faithfulness gates meaningful and cheap.

`GENERATOR_MODE=pulseplate_runtime` tries to execute the real PulsePlate runtime path with an evaluation stub provider.


In [ ]:
SENTENCE_RE = re.compile(r"(?<=[.!?。！？])\s+|\n+")

def _first_useful_sentence(text: str, *, min_chars: int = 40, max_chars: int = 420) -> str:
    for sentence in SENTENCE_RE.split(text):
        s = re.sub(r"\s+", " ", sentence).strip()
        if len(s) >= min_chars:
            return s[:max_chars]
    return re.sub(r"\s+", " ", text).strip()[:max_chars]

def _confidence_from_retrieval(retrieved: list[dict[str, Any]]) -> float:
    if not retrieved:
        return 0.05
    raw = float(retrieved[0].get("retrieval_score") or 0.0)
    # TF-IDF cosine is already roughly [0,1], but clamp for calibration stability.
    return float(np.clip(raw, 0.01, 0.99))

def extractive_grounded_generate(query: str, retrieved: list[dict[str, Any]]) -> tuple[str, float, dict[str, Any]]:
    if not retrieved:
        return (
            "Недостаточно подтверждённого контекста для ответа. Требуется эскалация.",
            0.05,
            {"generator": "extractive_stub", "reason": "no_retrieved_context"},
        )
    top = retrieved[0]
    evidence = _first_useful_sentence(str(top.get("doc_snippet", "")))
    answer = (
        f"На основе найденного источника `{top.get('source_url')}`: "
        f"\"{evidence}\"\n\n"
        f"Ответ на запрос: {query}\n"
        "Ограничение: это evidence-grounded offline evaluation response, не медицинская рекомендация."
    )
    return (
        answer,
        _confidence_from_retrieval(retrieved),
        {"generator": "extractive_stub", "evidence_source": top.get("source_url")},
    )

class EvalStubProvider:
    name = "pulseplate_eval_stub"

    def __init__(self, retrieved: list[dict[str, Any]]) -> None:
        self.retrieved = retrieved

    async def generate(self, text: str) -> str:
        answer, _, _ = extractive_grounded_generate(text, self.retrieved)
        return answer

async def pulseplate_runtime_generate(
    query: str,
    retrieved: list[dict[str, Any]],
    *,
    user_tier: str,
    subject_id: int | None,
) -> tuple[str, float, dict[str, Any]]:
    prepare_insight_runtime = PULSEPLATE_IMPORTS.get("prepare_insight_runtime")
    generate_traced_insight = PULSEPLATE_IMPORTS.get("generate_traced_insight")
    if prepare_insight_runtime is None or generate_traced_insight is None:
        raise RuntimeError("PulsePlate runtime imports unavailable")

    provider = EvalStubProvider(retrieved)
    prepared = prepare_insight_runtime(
        text=query,
        use_rag=True,
        philosophy_router_enabled=os.getenv("FEATURE_PHILOSOPHY_ROUTER", "false").lower() in {"1", "true", "on", "yes"},
        philosophy_linguistic_enabled=os.getenv("FEATURE_PHILOSOPHY_LINGUISTIC", "false").lower() in {"1", "true", "on", "yes"},
        provider_loader=lambda: provider,
        transparency_loader=lambda: (
            "ai_generated_insight",
            "Educational wellness information only; no diagnosis or treatment.",
        ),
        direct_provider_factory=lambda: provider,
    )
    result = await generate_traced_insight(
        runtime=prepared.runtime,
        text=query,
        lang=None,
        provider=prepared.provider,
        use_rag=True,
        philo_validation_enabled=True,
        recursive_rag_enabled=False,
        subject_id=subject_id,
        philosophy_router_enabled=False,
        philosophy_phase12_enabled=False,
        philosophy_linguistic_enabled=False,
        philosophy_pragmatic_enabled=False,
        route_path="/api/v1/insight",
        route_type=prepared.decision.route_type.value,
        user_tier=user_tier,
    )
    answer = str(getattr(result, "insight", ""))
    confidence = getattr(result, "confidence", None)
    if confidence is None:
        confidence = _confidence_from_retrieval(retrieved)
    return (
        answer,
        float(np.clip(float(confidence), 0.01, 0.99)),
        {
            "generator": "pulseplate_runtime_with_eval_stub_provider",
            "provider_name": getattr(result, "provider_name", None),
            "rag_used": getattr(result, "rag_used", None),
            "hops": getattr(result, "hops", None),
            "latency_ms": getattr(result, "latency_ms", None),
        },
    )

async def generate_answer(
    query: str,
    retrieved: list[dict[str, Any]],
    *,
    user_tier: str,
    subject_id: int | None,
) -> tuple[str, float, dict[str, Any]]:
    if GENERATOR_MODE == "pulseplate_runtime":
        try:
            return await pulseplate_runtime_generate(
                query,
                retrieved,
                user_tier=user_tier,
                subject_id=subject_id,
            )
        except Exception as exc:
            print(f"[WARN] PulsePlate runtime generator failed; falling back to extractive stub: {type(exc).__name__}: {exc}")
    return extractive_grounded_generate(query, retrieved)


## 5. Faithfulness utilities: claims, exact evidence, NLI fallback

The NLI scorer uses `transformers` only when `ENABLE_NLI_MODEL=true`. Otherwise it uses a deterministic lexical support proxy, which is suitable for smoke tests but not enough for a final release gate.


In [ ]:
def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower()).strip()

def tokenize(text: str) -> list[str]:
    return re.findall(r"[\wА-Яа-яЁё]+", normalize_text(text), flags=re.UNICODE)

def extract_claims(answer: str, *, max_claims: int = 8) -> list[dict[str, Any]]:
    claims: list[dict[str, Any]] = []
    cursor = 0
    for part in SENTENCE_RE.split(answer):
        span = part.strip()
        if len(span) < 20:
            cursor += len(part) + 1
            continue
        start = answer.find(span, cursor)
        end = start + len(span) if start >= 0 else -1
        claims.append({"span_text": span, "start": start, "end": end})
        cursor = max(cursor, end)
        if len(claims) >= max_claims:
            break
    return claims

def answer_contains_exact_evidence(answer: str, contexts: list[str], *, min_tokens: int = 8) -> bool:
    context_norm = "\n".join(normalize_text(c) for c in contexts)

    # Quoted evidence is the strongest exact-match signal.
    for quoted in re.findall(r'"([^"]{20,})"', answer):
        q_norm = normalize_text(quoted)
        if q_norm and q_norm in context_norm:
            return True

    # Fallback: any answer sentence copied from retrieved context.
    for sentence in SENTENCE_RE.split(answer):
        toks = tokenize(sentence)
        if len(toks) < min_tokens:
            continue
        s_norm = normalize_text(sentence)
        if s_norm and s_norm in context_norm:
            return True

    # N-gram copy check.
    toks = tokenize(answer)
    for n in range(min_tokens, min(len(toks), 18) + 1):
        for i in range(0, max(0, len(toks) - n + 1)):
            phrase = " ".join(toks[i : i + n])
            if phrase in context_norm:
                return True
    return False

def lexical_support_score(claim: str, contexts: list[str]) -> float:
    claim_tokens = set(tokenize(claim))
    if not claim_tokens:
        return 0.0
    best = 0.0
    for ctx in contexts:
        ctx_tokens = set(tokenize(ctx))
        if not ctx_tokens:
            continue
        overlap = len(claim_tokens & ctx_tokens) / max(1, len(claim_tokens))
        best = max(best, overlap)
    return float(np.clip(best, 0.0, 1.0))

_NLI_PIPELINE = None

def load_nli_pipeline():
    global _NLI_PIPELINE
    if _NLI_PIPELINE is not None:
        return _NLI_PIPELINE
    if not ENABLE_NLI_MODEL:
        return None
    try:
        from transformers import pipeline
        _NLI_PIPELINE = pipeline(
            "text-classification",
            model=NLI_MODEL_NAME,
            tokenizer=NLI_MODEL_NAME,
            truncation=True,
        )
        return _NLI_PIPELINE
    except Exception as exc:
        print(f"[WARN] NLI model unavailable; lexical support fallback will be used: {type(exc).__name__}: {exc}")
        return None

def nli_entailment_score(claim: str, contexts: list[str]) -> float:
    nli = load_nli_pipeline()
    if nli is None:
        return lexical_support_score(claim, contexts)
    premise = "\n\n".join(contexts[:8])[:3500]
    if not premise.strip() or not claim.strip():
        return 0.0
    try:
        outputs = nli({"text": premise, "text_pair": claim}, top_k=None)
    except Exception:
        outputs = nli(f"{premise}\n\nHypothesis: {claim}", top_k=None)

    if isinstance(outputs, dict):
        outputs = [outputs]
    best = 0.0
    for item in outputs:
        label = str(item.get("label", "")).lower()
        score = float(item.get("score", 0.0))
        if "entail" in label:
            best = max(best, score)
    return float(np.clip(best, 0.0, 1.0))

def evaluate_faithfulness(answer: str, retrieved: list[dict[str, Any]]) -> dict[str, Any]:
    contexts = [str(r.get("doc_snippet", "")) for r in retrieved]
    claims = extract_claims(answer)
    entailments = []
    support_flags = []
    for claim in claims:
        score = nli_entailment_score(claim["span_text"], contexts)
        entailments.append(score)
        support_flags.append(score >= SUPPORT_ENTAILMENT_THRESHOLD)

    return {
        "extracted_claim_spans": claims,
        "per_span_entailment_score": entailments,
        "support_flags": support_flags,
        "evidence_exact_match": answer_contains_exact_evidence(answer, contexts),
        "mean_nli_entailment": float(np.mean(entailments)) if entailments else 0.0,
        "support_precision": float(np.mean(support_flags)) if support_flags else 0.0,
    }

def validate_output_with_pulseplate(answer: str) -> dict[str, Any]:
    validate_llm_output = PULSEPLATE_IMPORTS.get("validate_llm_output")
    if validate_llm_output is None:
        return {"philosophy_validator_available": False, "ok": None, "blockers": []}
    try:
        report = validate_llm_output(answer, domain="rag_eval")
        blockers = [
            {
                "code": getattr(b, "code", None),
                "start": getattr(b, "start", None),
                "end": getattr(b, "end", None),
                "matched": getattr(b, "matched", None),
            }
            for b in getattr(report, "blockers", [])
        ]
        return {
            "philosophy_validator_available": True,
            "ok": bool(getattr(report, "ok", False)),
            "blockers": blockers,
        }
    except Exception as exc:
        return {
            "philosophy_validator_available": True,
            "ok": None,
            "blockers": [],
            "error": f"{type(exc).__name__}: {exc}",
        }


## 6. Retrieval metrics

Matching is path-fragment tolerant because PulsePlate chunks use provenance fields such as `file`, `chunk_id`, and `source_url`.


In [ ]:
def _doc_matches(doc_id: str, source_url: str, gold: str) -> bool:
    d = normalize_text(doc_id)
    s = normalize_text(source_url)
    g = normalize_text(gold)
    if not g:
        return False
    return g in d or g in s or d in g or s in g

def relevance_vector(retrieved: list[dict[str, Any]], gold_doc_ids: list[str], *, k: int) -> list[int]:
    rel = []
    for r in retrieved[:k]:
        doc_id = str(r.get("doc_id", ""))
        source_url = str(r.get("source_url", ""))
        rel.append(1 if any(_doc_matches(doc_id, source_url, g) for g in gold_doc_ids) else 0)
    return rel

def recall_at_k(retrieved: list[dict[str, Any]], gold_doc_ids: list[str], *, k: int) -> float:
    if not gold_doc_ids:
        return float("nan")
    found = set()
    for r in retrieved[:k]:
        for g in gold_doc_ids:
            if _doc_matches(str(r.get("doc_id", "")), str(r.get("source_url", "")), g):
                found.add(g)
    return len(found) / max(1, len(set(gold_doc_ids)))

def mrr_at_k(retrieved: list[dict[str, Any]], gold_doc_ids: list[str], *, k: int) -> float:
    if not gold_doc_ids:
        return float("nan")
    for idx, r in enumerate(retrieved[:k], start=1):
        if any(_doc_matches(str(r.get("doc_id", "")), str(r.get("source_url", "")), g) for g in gold_doc_ids):
            return 1.0 / idx
    return 0.0

def ndcg_at_k(retrieved: list[dict[str, Any]], gold_doc_ids: list[str], *, k: int) -> float:
    if not gold_doc_ids:
        return float("nan")
    rel = relevance_vector(retrieved, gold_doc_ids, k=k)
    dcg = sum(v / math.log2(i + 2) for i, v in enumerate(rel))
    ideal_relevant = min(len(set(gold_doc_ids)), k)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(ideal_relevant))
    return dcg / idcg if idcg > 0 else 0.0


## 7. Main evaluation loop

This produces one trace per query using the requested JSONL/Parquet-friendly shape.


In [ ]:
def stable_hash(value: Any) -> str:
    raw = json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:16]

def coerce_subject_id(value: Any) -> int | None:
    if value is None:
        return None
    if isinstance(value, float) and math.isnan(value):
        return None
    try:
        return int(value)
    except Exception:
        return None

async def evaluate_one(row: pd.Series) -> dict[str, Any]:
    query_text = str(row["query_text"])
    trace_id = f"{EXPERIMENT_ID}:{row['query_id']}"
    started = time.perf_counter()

    guard_result = {"available": False, "is_safe": True, "threats": []}
    scan_ai_agent_input = PULSEPLATE_IMPORTS.get("scan_ai_agent_input")
    if scan_ai_agent_input is not None:
        try:
            scan = scan_ai_agent_input(query_text)
            guard_result = {
                "available": True,
                "is_safe": bool(getattr(scan, "is_safe", False)),
                "threats": [
                    {
                        "category": getattr(t, "category", None),
                        "severity": getattr(t, "severity", None),
                        "reason": getattr(t, "reason", None),
                    }
                    for t in getattr(scan, "threats", [])
                ],
            }
        except Exception as exc:
            guard_result = {
                "available": True,
                "is_safe": False,
                "threats": [{"category": "guard_error", "severity": "critical", "reason": str(exc)}],
            }

    subject_id = coerce_subject_id(row.get("subject_id"))
    if not guard_result["is_safe"]:
        latency_ms = int((time.perf_counter() - started) * 1000)
        return {
            "trace_id": trace_id,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "experiment_id": EXPERIMENT_ID,
            "query_text": query_text,
            "user_context_hash": stable_hash({"subject_id": subject_id, "user_tier": row.get("user_tier")}),
            "top_k_retrieved": [],
            "generator_output": "",
            "extracted_claim_spans": [],
            "per_span_entailment_score": [],
            "support_flags": [],
            "generator_logprob": None,
            "confidence": 0.0,
            "post_hoc_calibrated_confidence": None,
            "routing_decision": "blocked_by_agent_input_guard",
            "latency": latency_ms,
            "human_label_if_any": row.get("human_label_if_any"),
            "gold_doc_ids": row.get("gold_doc_ids", []),
            "gold_answer": row.get("gold_answer", ""),
            "expected_claims": row.get("expected_claims", []),
            "evidence_quotes": row.get("evidence_quotes", []),
            "agent_input_guard": guard_result,
            "philosophy_output_validation": None,
            "retrieval_metrics": {},
            "faithfulness_metrics": {},
            "generator_metadata": {},
        }

    retrieved, retrieval_stats = await retrieve(query_text, top_k=TOP_K, subject_id=subject_id)
    answer, confidence, gen_meta = await generate_answer(
        query_text,
        retrieved,
        user_tier=str(row.get("user_tier") or "PRO"),
        subject_id=subject_id,
    )
    faith = evaluate_faithfulness(answer, retrieved)
    output_validation = validate_output_with_pulseplate(answer)

    latency_ms = int((time.perf_counter() - started) * 1000)
    gold_doc_ids = row.get("gold_doc_ids", [])
    retrieval_metrics = {
        "recall@3": recall_at_k(retrieved, gold_doc_ids, k=3),
        "recall@10": recall_at_k(retrieved, gold_doc_ids, k=10),
        "recall_at_effective_k": recall_at_k(retrieved, gold_doc_ids, k=len(retrieved) or TOP_K),
        "mrr@10": mrr_at_k(retrieved, gold_doc_ids, k=10),
        "ndcg@10": ndcg_at_k(retrieved, gold_doc_ids, k=10),
    }

    return {
        "trace_id": trace_id,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "experiment_id": EXPERIMENT_ID,
        "query_text": query_text,
        "user_context_hash": stable_hash({"subject_id": subject_id, "user_tier": row.get("user_tier")}),
        "top_k_retrieved": retrieved,
        "retrieval_stats": retrieval_stats,
        "generator_output": answer,
        "extracted_claim_spans": faith["extracted_claim_spans"],
        "per_span_entailment_score": faith["per_span_entailment_score"],
        "support_flags": faith["support_flags"],
        "generator_logprob": None,
        "confidence": confidence,
        "post_hoc_calibrated_confidence": None,
        "routing_decision": "pending_calibration",
        "latency": latency_ms,
        "human_label_if_any": row.get("human_label_if_any"),
        "gold_doc_ids": gold_doc_ids,
        "gold_answer": row.get("gold_answer", ""),
        "expected_claims": row.get("expected_claims", []),
        "evidence_quotes": row.get("evidence_quotes", []),
        "agent_input_guard": guard_result,
        "philosophy_output_validation": output_validation,
        "retrieval_metrics": retrieval_metrics,
        "faithfulness_metrics": {
            "evidence_exact_match": faith["evidence_exact_match"],
            "mean_nli_entailment": faith["mean_nli_entailment"],
            "support_precision": faith["support_precision"],
        },
        "generator_metadata": gen_meta,
    }

async def run_evaluation(df: pd.DataFrame) -> list[dict[str, Any]]:
    traces: list[dict[str, Any]] = []
    for i, (_, row) in enumerate(df.iterrows(), start=1):
        traces.append(await evaluate_one(row))
        if i % 25 == 0 or i == len(df):
            print(f"evaluated {i}/{len(df)}")
    return traces

traces = await run_evaluation(eval_df)
traces_df = pd.DataFrame(traces)
display(traces_df[["trace_id", "confidence", "routing_decision", "latency"]].head())


## 8. Calibration: temperature scaling, ECE, Brier

Calibration labels are taken from `human_label_if_any` when present. If absent, the notebook derives a conservative proxy from faithfulness: exact evidence + support precision + mean entailment.


In [ ]:
def proxy_correctness(trace: dict[str, Any]) -> int:
    label = trace.get("human_label_if_any")
    if label is not None and not (isinstance(label, float) and math.isnan(label)):
        try:
            return int(float(label) >= 0.5)
        except Exception:
            pass
    faith = trace.get("faithfulness_metrics", {}) or {}
    output_validation = trace.get("philosophy_output_validation") or {}
    safe_output = output_validation.get("ok")
    if safe_output is False:
        return 0
    return int(
        bool(faith.get("evidence_exact_match", False))
        and float(faith.get("support_precision", 0.0)) >= GATE_THRESHOLDS["support_precision"]
        and float(faith.get("mean_nli_entailment", 0.0)) >= min(0.75, GATE_THRESHOLDS["mean_nli_entailment"])
    )

def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))

def logit(p: np.ndarray) -> np.ndarray:
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def nll_binary(y: np.ndarray, p: np.ndarray) -> float:
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())

def temperature_scale_confidence(conf: Sequence[float], y: Sequence[int]) -> tuple[float, np.ndarray]:
    conf_arr = np.clip(np.array(conf, dtype=float), 1e-6, 1 - 1e-6)
    y_arr = np.array(y, dtype=float)
    logits = logit(conf_arr)

    # Robust grid search; avoids requiring scipy.
    candidates = np.linspace(0.25, 5.0, 400)
    losses = []
    for t in candidates:
        scaled = sigmoid(logits / t)
        losses.append(nll_binary(y_arr, scaled))
    best_t = float(candidates[int(np.argmin(losses))])
    return best_t, sigmoid(logits / best_t)

def expected_calibration_error(y: Sequence[int], p: Sequence[float], *, n_bins: int = 10) -> float:
    y_arr = np.array(y, dtype=float)
    p_arr = np.array(p, dtype=float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (p_arr >= lo) & (p_arr < hi if hi < 1.0 else p_arr <= hi)
        if not mask.any():
            continue
        acc = y_arr[mask].mean()
        conf = p_arr[mask].mean()
        ece += (mask.mean()) * abs(acc - conf)
    return float(ece)

def brier_score(y: Sequence[int], p: Sequence[float]) -> float:
    y_arr = np.array(y, dtype=float)
    p_arr = np.array(p, dtype=float)
    return float(np.mean((p_arr - y_arr) ** 2))

labels = [proxy_correctness(t) for t in traces]
raw_conf = [float(t.get("confidence") or 0.0) for t in traces]
temperature, calibrated_conf = temperature_scale_confidence(raw_conf, labels)

for trace, p_cal in zip(traces, calibrated_conf):
    trace["post_hoc_calibrated_confidence"] = float(p_cal)
    if trace.get("routing_decision") == "blocked_by_agent_input_guard":
        continue
    faith = trace.get("faithfulness_metrics", {}) or {}
    should_escalate = (
        float(p_cal) < ROUTING_CONFIDENCE_THRESHOLD
        or not faith.get("evidence_exact_match", False)
        or float(faith.get("support_precision", 0.0)) < GATE_THRESHOLDS["support_precision"]
    )
    trace["routing_decision"] = "escalate" if should_escalate else "ship_candidate"

calibration_metrics = {
    "temperature": temperature,
    "ece": expected_calibration_error(labels, calibrated_conf),
    "brier": brier_score(labels, calibrated_conf),
    "mean_raw_confidence": float(np.mean(raw_conf)) if raw_conf else 0.0,
    "mean_calibrated_confidence": float(np.mean(calibrated_conf)) if len(calibrated_conf) else 0.0,
}

print(calibration_metrics)
pd.DataFrame(
    {
        "label": labels,
        "raw_confidence": raw_conf,
        "calibrated_confidence": calibrated_conf,
    }
).head()


## 9. Optional RAGAS block

This block is skipped unless `ragas` and `datasets` are installed and the environment has the required model/embedding configuration.


In [ ]:
def maybe_run_ragas(traces: list[dict[str, Any]]) -> pd.DataFrame | None:
    try:
        from datasets import Dataset
        from ragas import evaluate
        from ragas.metrics import context_precision, context_recall, faithfulness
    except Exception as exc:
        print(f"RAGAS unavailable; skipped: {type(exc).__name__}: {exc}")
        return None

    rows = []
    for t in traces:
        rows.append(
            {
                "question": t["query_text"],
                "answer": t["generator_output"],
                "contexts": [r.get("doc_snippet", "") for r in t.get("top_k_retrieved", [])[:10]],
                "ground_truth": t.get("gold_answer", ""),
            }
        )
    dataset = Dataset.from_list(rows)
    try:
        result = evaluate(
            dataset,
            metrics=[faithfulness, context_precision, context_recall],
        )
        return result.to_pandas()
    except Exception as exc:
        print(f"RAGAS evaluation failed; skipped: {type(exc).__name__}: {exc}")
        return None

ragas_df = maybe_run_ragas(traces)
if ragas_df is not None:
    display(ragas_df.head())


## 10. Aggregate metrics and release gates


In [ ]:
def nanmean(values: Iterable[float]) -> float:
    arr = np.array([v for v in values if v is not None and not math.isnan(float(v))], dtype=float)
    return float(arr.mean()) if len(arr) else float("nan")

retrieval_summary = {
    "recall@3": nanmean(t["retrieval_metrics"].get("recall@3", float("nan")) for t in traces),
    "recall@10": nanmean(t["retrieval_metrics"].get("recall@10", float("nan")) for t in traces),
    "recall_at_effective_k": nanmean(t["retrieval_metrics"].get("recall_at_effective_k", float("nan")) for t in traces),
    "mrr@10": nanmean(t["retrieval_metrics"].get("mrr@10", float("nan")) for t in traces),
    "ndcg@10": nanmean(t["retrieval_metrics"].get("ndcg@10", float("nan")) for t in traces),
}

faithfulness_summary = {
    "evidence_exact_match_rate": float(np.mean([t["faithfulness_metrics"].get("evidence_exact_match", False) for t in traces])),
    "mean_nli_entailment": float(np.mean([t["faithfulness_metrics"].get("mean_nli_entailment", 0.0) for t in traces])),
    "support_precision": float(np.mean([t["faithfulness_metrics"].get("support_precision", 0.0) for t in traces])),
}

routing_summary = {
    "escalation_rate": float(np.mean([t["routing_decision"] == "escalate" for t in traces])),
    "blocked_by_guard_rate": float(np.mean([t["routing_decision"] == "blocked_by_agent_input_guard" for t in traces])),
    "ship_candidate_rate": float(np.mean([t["routing_decision"] == "ship_candidate" for t in traces])),
}

metrics_summary = {
    "experiment_id": EXPERIMENT_ID,
    "sample_size": len(traces),
    "retriever_mode": RETRIEVER_MODE,
    "generator_mode": GENERATOR_MODE,
    "retrieval": retrieval_summary,
    "faithfulness": faithfulness_summary,
    "calibration": calibration_metrics,
    "routing": routing_summary,
    "thresholds": GATE_THRESHOLDS,
}

gate_checks = {
    "Gate A: Recall@effective_k": retrieval_summary["recall_at_effective_k"] >= GATE_THRESHOLDS["recall_at_50"]
        if not math.isnan(retrieval_summary["recall_at_effective_k"]) else False,
    "Gate B1: Evidence exact-match": faithfulness_summary["evidence_exact_match_rate"] >= GATE_THRESHOLDS["evidence_exact_match_rate"],
    "Gate B2: Mean NLI entailment": faithfulness_summary["mean_nli_entailment"] >= GATE_THRESHOLDS["mean_nli_entailment"],
    "Gate B3: Support precision": faithfulness_summary["support_precision"] >= GATE_THRESHOLDS["support_precision"],
    "Gate C1: ECE": calibration_metrics["ece"] <= GATE_THRESHOLDS["ece"],
    "Gate C2: Escalation corridor": GATE_THRESHOLDS["escalation_min"] <= routing_summary["escalation_rate"] <= GATE_THRESHOLDS["escalation_max"],
}
release_decision = "PASS" if all(gate_checks.values()) else "NO-GO"

display(pd.DataFrame(
    [
        {"metric": "Recall@effective_k", "value": retrieval_summary["recall_at_effective_k"], "threshold": GATE_THRESHOLDS["recall_at_50"]},
        {"metric": "Evidence exact-match rate", "value": faithfulness_summary["evidence_exact_match_rate"], "threshold": GATE_THRESHOLDS["evidence_exact_match_rate"]},
        {"metric": "Mean entailment", "value": faithfulness_summary["mean_nli_entailment"], "threshold": GATE_THRESHOLDS["mean_nli_entailment"]},
        {"metric": "Support precision", "value": faithfulness_summary["support_precision"], "threshold": GATE_THRESHOLDS["support_precision"]},
        {"metric": "ECE", "value": calibration_metrics["ece"], "threshold": GATE_THRESHOLDS["ece"]},
        {"metric": "Escalation rate", "value": routing_summary["escalation_rate"], "threshold": f"{GATE_THRESHOLDS['escalation_min']}..{GATE_THRESHOLDS['escalation_max']}"},
    ]
))

display(pd.DataFrame([{"gate": k, "pass": v} for k, v in gate_checks.items()]))
print("Release decision:", release_decision)


## 11. Export JSONL + Parquet + Markdown report

Output paths are under `artifacts/rag_eval/<experiment_id>/` by default, matching PulsePlate's local-artifact policy.


In [ ]:
def json_default(obj: Any) -> Any:
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, Path):
        return str(obj)
    return str(obj)

traces_jsonl_path = RUN_DIR / "traces.jsonl"
with traces_jsonl_path.open("w", encoding="utf-8") as f:
    for trace in traces:
        f.write(json.dumps(trace, ensure_ascii=False, default=json_default) + "\n")

# Flatten for analytics-friendly Parquet.
flat_rows = []
for t in traces:
    flat_rows.append(
        {
            "trace_id": t["trace_id"],
            "timestamp": t["timestamp"],
            "experiment_id": t["experiment_id"],
            "query_text": t["query_text"],
            "confidence": t["confidence"],
            "post_hoc_calibrated_confidence": t["post_hoc_calibrated_confidence"],
            "routing_decision": t["routing_decision"],
            "latency": t["latency"],
            "human_label_if_any": t["human_label_if_any"],
            "recall_at_3": t["retrieval_metrics"].get("recall@3"),
            "recall_at_10": t["retrieval_metrics"].get("recall@10"),
            "recall_at_effective_k": t["retrieval_metrics"].get("recall_at_effective_k"),
            "mrr_at_10": t["retrieval_metrics"].get("mrr@10"),
            "ndcg_at_10": t["retrieval_metrics"].get("ndcg@10"),
            "evidence_exact_match": t["faithfulness_metrics"].get("evidence_exact_match"),
            "mean_nli_entailment": t["faithfulness_metrics"].get("mean_nli_entailment"),
            "support_precision": t["faithfulness_metrics"].get("support_precision"),
            "top_doc_id": (t["top_k_retrieved"][0]["doc_id"] if t["top_k_retrieved"] else None),
            "top_source_url": (t["top_k_retrieved"][0]["source_url"] if t["top_k_retrieved"] else None),
        }
    )
flat_df = pd.DataFrame(flat_rows)

parquet_path = RUN_DIR / "traces.parquet"
csv_fallback_path = RUN_DIR / "traces.csv"
try:
    flat_df.to_parquet(parquet_path, index=False)
    parquet_status = str(parquet_path)
except Exception as exc:
    flat_df.to_csv(csv_fallback_path, index=False)
    parquet_status = f"Parquet unavailable ({type(exc).__name__}: {exc}); wrote CSV fallback: {csv_fallback_path}"

metrics_path = RUN_DIR / "metrics_summary.json"
metrics_path.write_text(
    json.dumps(
        {**metrics_summary, "gate_checks": gate_checks, "release_decision": release_decision},
        ensure_ascii=False,
        indent=2,
        default=json_default,
    ),
    encoding="utf-8",
)

report_path = RUN_DIR / "gate_report.md"
report_path.write_text(
    "\n".join(
        [
            f"# PulsePlate RAG Release Gate Report — {EXPERIMENT_ID}",
            "",
            f"- Decision: **{release_decision}**",
            f"- Sample size: `{len(traces)}`",
            f"- Retriever mode: `{RETRIEVER_MODE}`",
            f"- Generator mode: `{GENERATOR_MODE}`",
            "",
            "## Gate checks",
            *[f"- [{'x' if passed else ' '}] {name}" for name, passed in gate_checks.items()],
            "",
            "## Retrieval",
            *[f"- `{k}`: `{v}`" for k, v in retrieval_summary.items()],
            "",
            "## Faithfulness",
            *[f"- `{k}`: `{v}`" for k, v in faithfulness_summary.items()],
            "",
            "## Calibration",
            *[f"- `{k}`: `{v}`" for k, v in calibration_metrics.items()],
            "",
            "## Routing",
            *[f"- `{k}`: `{v}`" for k, v in routing_summary.items()],
            "",
            "## Artifacts",
            f"- JSONL: `{traces_jsonl_path}`",
            f"- Parquet/CSV: `{parquet_status}`",
            f"- Metrics: `{metrics_path}`",
        ]
    ),
    encoding="utf-8",
)

print("Wrote:")
print(" ", traces_jsonl_path)
print(" ", parquet_status)
print(" ", metrics_path)
print(" ", report_path)


## 12. Inspect failures

Use this table to drill into no-go cases before opening a remediation PR or changing RAG/provider behavior.


In [ ]:
failure_rows = []
for t in traces:
    if t["routing_decision"] != "ship_candidate":
        failure_rows.append(
            {
                "trace_id": t["trace_id"],
                "query_text": t["query_text"],
                "routing_decision": t["routing_decision"],
                "confidence": t["confidence"],
                "calibrated_confidence": t["post_hoc_calibrated_confidence"],
                "evidence_exact_match": t["faithfulness_metrics"].get("evidence_exact_match"),
                "mean_nli_entailment": t["faithfulness_metrics"].get("mean_nli_entailment"),
                "support_precision": t["faithfulness_metrics"].get("support_precision"),
                "top_source_url": t["top_k_retrieved"][0]["source_url"] if t["top_k_retrieved"] else None,
                "generator_output": t["generator_output"][:500],
            }
        )

failures_df = pd.DataFrame(failure_rows)
display(failures_df.head(30))


## 13. CI / weekly usage

Suggested local command from the repository root:

```bash
PULSEPLATE_RAG_EVAL_INPUT=data/evals/rag_weekly_500.jsonl \
RETRIEVER_MODE=local_tfidf \
GENERATOR_MODE=extractive_stub \
jupyter nbconvert --to notebook --execute notebooks/pulseplate_rag_release_gates.ipynb \
  --output artifacts/rag_eval/latest_executed.ipynb
```

For a stricter weekly lane inside a configured PulsePlate environment:

```bash
PULSEPLATE_RAG_EVAL_INPUT=data/evals/rag_weekly_500.jsonl \
RETRIEVER_MODE=pulseplate \
GENERATOR_MODE=pulseplate_runtime \
ENABLE_NLI_MODEL=true \
NLI_MODEL_NAME=roberta-large-mnli \
jupyter nbconvert --to notebook --execute notebooks/pulseplate_rag_release_gates.ipynb \
  --output artifacts/rag_eval/latest_executed.ipynb
```

Do not commit `artifacts/rag_eval/*`; keep it as local/CI artifact storage.
